# 07 — the dark bound, and the stack for `eta_comb`

Session 03, capturing to `protocols/03-dark-bound.md`. No light source, cap on,
about five hours at gain 250, offset 15, -10 C.

**What this notebook is for.** Capturing one night of darks and interleaved bias,
and measuring three things from them: an upper bound on dark current `D` at -10 C;
an upper bound on the combination efficiency `eta_comb` from a stack that needs no
registration (L15); and whether the dark carries spatial structure — glow, or
dark-signal non-uniformity — inside the ROI the model will use.

**What it is not for.** It is not a `D(T)` sweep: MISSION fixes the setpoint at
-10 C, so temperature is not an axis here and no doubling temperature is fitted
(L14 is explicit that there is nothing to fit one to). It does not integrate
anything — stacking is a PixInsight step and belongs to build step 5; tonight
captures. And it does not touch the archive: every frame here is shot to a
protocol, so its header is trusted in full.

**The result this is designed to produce is a bound.** The best outcome is that
the slope does not exceed its own uncertainty, `D` leaves `sigma^2` in the model,
and the temperature axis never opens. An upper bound is a result and is recorded
as one.

**The error bar is the pedestal, not the frame count.** Rule 2 of the protocol:
the statistical floor is 1.3e-6 e-/px/s at 600 s, four decades below the number
being tested, so nothing here is limited by how many frames it shoots. What limits
it is pedestal wander — 11.7 counts across a bracket would fake `D = 0.01 e-/px/s`.
That is why the bias blocks are interleaved on a 20-minute clock rather than
batched, and why they are a published result in their own right.

The explaining half is `08`.

## The plan, before the camera is opened

Every number here is measured, not fitted: gain 250 is a swept point in both
session 01 and session 02, which is why the protocol moved off 252 (session 02's
gain law has a 1.34% residual against its own 1% rule, so `g` is not interpolable).

In [ ]:
import datetime as dt
import json
import pathlib
import sys
import time

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, fits as F, spatial as SP, stats as ST

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session03"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

GAIN = 250              # a swept point in sessions 01 and 02; 252 is not
OFFSET = 15             # results/bias_constants.json, project_offset
ROI = (1408, 568, 1024, 1024)          # as session 01
FULL_FRAME = (0, 0, 3840, 2160)
SETPOINT_C = asi.SETPOINT_C

FRAME_GAP_S = 0.2       # readout, USB and the file write are what heat the sensor
HOLD_TIMEOUT_S = 600.0  # a hold that never ends is a cooler fault, not a wait
MAX_RETAKES = 3         # a retaken 600 s dark costs ten minutes; three is the budget

# Session 01 and 02, at exactly this gain and offset.  Predictions for the
# pre-flight to check against, never inputs to anything published tonight.
bias_c = json.loads((RESULTS / "bias_constants.json").read_text())
ptc_c = json.loads((RESULTS / "ptc_constants.json").read_text())
G_E_PER_COUNT = ptc_c["system_gain"]["value"]["250"]
G_ERR = ptc_c["system_gain"]["uncertainty"]["250"]
PEDESTAL_PRED = 76.66   # results/bias_sweep.csv, gain 250 at offset 15
R_COUNTS = 1.7281       # ditto, R_sd

print(f"gain {GAIN}, offset {OFFSET}, ROI {ROI}, setpoint {SETPOINT_C} C")
print(f"g = {G_E_PER_COUNT} +/- {G_ERR} e-/count   "
      f"(one count is {G_E_PER_COUNT:.3f} e-, which is the quantisation L14 lost to)")
print(f"predicted pedestal {PEDESTAL_PRED} counts, R {R_COUNTS} counts")

plane_px = ROI[2] * ROI[3] // 4
stat_floor = np.sqrt(2) * R_COUNTS / np.sqrt(plane_px * 10) * G_E_PER_COUNT / 600
print(f"\nstatistical floor on D at 600 s: {stat_floor:.2e} e-/px/s")
print(f"pedestal wander that would fake D = 0.01 e-/px/s: "
      f"{0.01 * 600 / G_E_PER_COUNT:.1f} counts")
print(f"                          ... and 1e-4 e-/px/s: "
      f"{1e-4 * 600 / G_E_PER_COUNT:.2f} counts")

## The schedule

One list of blocks, in execution order. The interleave is on a 20-minute clock:
every dark block is bracketed by a bias block before and after, and no bracket
spans more than 20 minutes of wall clock. The block index is in the filename, so
the time order survives on disk even if a header is ever in doubt — but `DATE-OBS`
is what the analysis interpolates on, because that is the actual clock.

The last interleaved bias block of D4 *is* the protocol's B4; there is no separate
trailing block.

In [ ]:
N_BIAS = 10             # per block; sets the pedestal reference to 1.1e-3 counts
N_BIAS_FF = 5           # a full frame has 8x the pixels, so 5 beats the ROI's 10
BIAS = None             # resolved from the camera as its minimum exposure


def schedule(bias_s):
    """The night, as `(kind, exposure_s, n, roi)` blocks in execution order."""
    b = ("bias", bias_s, N_BIAS, ROI)
    ff_b = ("bias", bias_s, N_BIAS_FF, FULL_FRAME)
    blocks = [b,
              ("dark", 1.0, 20, ROI), b,        # D1
              ("dark", 60.0, 20, ROI), b]       # D2, one 20-minute bracket
    for _ in range(8):                          # D3: 32 darks at 300 s
        blocks += [("dark", 300.0, 4, ROI), b]
    for _ in range(4):                          # D4: 8 darks at 600 s
        blocks += [("dark", 600.0, 2, ROI), b]  # the last b is B4
    # The full-frame block is 40 minutes long and used to sit unbracketed at the
    # end of the night.  The 2026-08-31 smoke test measured the pedestal moving
    # 1.1 counts in ten minutes, so 40 minutes unbracketed is worth several
    # counts -- against the 0.12 that would fake D = 1e-4 e-/px/s.  It gets its
    # own bias either side, at its own ROI.
    blocks += [ff_b, ("dark", 600.0, 4, FULL_FRAME), ff_b]
    return blocks


plan = schedule(3.2e-05)          # the published minimum, for planning only
frames = sum(n for _, _, n, _ in plan)
seconds = sum(n * (e + FRAME_GAP_S + 0.4) for _, e, n, _ in plan)
gb = sum(n * r[2] * r[3] * 2 for _, _, n, r in plan) / 1e9

print(f"{len(plan)} blocks, {frames} frames, {gb:.2f} GB, "
      f"{seconds / 3600:.2f} h of capture")
for kind, e, n, r in plan:
    print(f"  {kind:5s} {e:7.4g} s x {n:2d}  {r[2]}x{r[3]}")

## Gate 1 — white balance, proved in the pixels

Nothing captured before this passes is usable (L01). Reading the control back only
proves the control took; the evidence is a modal step of 16 on all four planes.
Run on stored values, because the test goes vacuous in ADC counts.

In [ ]:
existing = list(FRAMES.glob("*.fits"))
if existing:
    print(f"WARNING: {len(existing)} frames already in {FRAMES}.\n"
          "A resumed night has a gap in the pedestal series.  That is recoverable\n"
          "— DATE-OBS says where — but the blocks either side of the gap are two\n"
          "brackets, not one, and 08 must treat them that way.")

rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=GAIN, offset=OFFSET)
BIAS = rig.min_exposure_s()               # measured, never assumed

for _ in range(2):                        # discards after the configuration change
    asi.capture(rig, BIAS)

gate1 = {}
for _ in range(5):
    mosaic, _ = asi.capture(rig, BIAS)
    for name, plane in SP.split(mosaic).items():     # stored values: the grid is 16 there
        gate1.setdefault(name, []).append(ST.value_step(plane))

for name in SP.PLANES:
    print(f"  {name:2s} modal step {gate1[name]}")
bad = {n: s for n, s in gate1.items() if set(s) != {16}}
assert not bad, (f"white balance is still being applied: {bad} -- stop the session, "
                 "nothing captured from here is usable (L01)")
print(f"\ngate 1 passed on five frames.  bias exposure {BIAS * 1e6:.0f} us")

## Gate 2 — the pedestal is where session 01 left it

A cheap check with a real failure behind it: if the pedestal at gain 250 does not
land near session 01's 76.66 counts, then either the gain or the offset is not what
this notebook thinks, and every subtraction tonight is against the wrong level.
Five counts is a wide band — this catches a wrong *setting*, not a drift.

In [ ]:
mosaic, hdr = asi.capture(rig, BIAS, imagetyp="BIAS")
levels = {n: float(ST.to_adc(p).mean()) for n, p in SP.split(mosaic).items()}
mean_level = float(np.mean(list(levels.values())))

for n, v in levels.items():
    print(f"  {n:2s} {v:8.3f} counts")
print(f"\nmean {mean_level:.3f} vs session 01's {PEDESTAL_PRED} "
      f"({mean_level - PEDESTAL_PRED:+.3f})")
print(f"header says gain {hdr['GAIN']}, offset {hdr['OFFSET']}")
assert abs(mean_level - PEDESTAL_PRED) < 5.0, (
    f"pedestal {mean_level:.2f} is not session 01's {PEDESTAL_PRED} at gain "
    f"{GAIN}/offset {OFFSET} -- check the settings before capturing anything")
assert hdr["GAIN"] == GAIN and hdr["OFFSET"] == OFFSET
print("gate 2 passed")

## Cool down, and log the curve

Ambient is assumed 25 C. Session 02 cooled from exactly 25.0 C and held -10 C on
65% duty with zero holds and zero retakes; tonight reads the sensor far less often,
and readout is what heats it, so the load is lighter than a run that already passed.
The duty at setpoint is the number to watch — it is the headroom this room leaves.

In [ ]:
AMBIENT_START_C = 25.0            # <- record the real reading before running

# Appended for the same reason `frames.csv` is: a resumed night has two
# cool-downs, and the first one is the one that describes the room.
cool_path = DATA / "cooldown.csv"
fresh = not cool_path.exists()
cool_log = open(cool_path, "a", newline="")
if fresh:
    cool_log.write("elapsed_s,temp_C,duty_pct\n")


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}\n")
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
duty = trace[-1][2]
print(f"\nsettled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C")
print(f"duty at setpoint {duty}%")
if duty > 85:
    print("  ^ under 15% headroom.  Five hours is a long time to hold that; "
          "check the fan and the ambient before starting.")

## Capture

The library does one frame; the loop is here (`CLAUDE.md`). Two rules the loop
enforces, both from the protocol:

- **Every frame outside the band is retaken rather than written.** A retaken 600 s
  dark costs ten minutes, so the budget is three; past that the block is recorded
  as it stands and the night is judged in `08` rather than silently patched.
- **Nothing changes between blocks except exposure.** Gain, offset and setpoint are
  set once, above. The ROI changes exactly once, for the full-frame block, after
  the ROI run has closed with its own bias block.

In [ ]:
log_path = DATA / "capture_log.txt"
log_file = open(log_path, "a", encoding="utf8")


def say(msg):
    print(msg, flush=True)
    log_file.write(msg + "\n")
    log_file.flush()            # data/ is gitignored and this is the session record


def in_band(header):
    t = header["CCD-TEMP"]
    return t is not None and abs(t - SETPOINT_C) <= asi.BAND_C


def capture_block(index, kind, exposure_s, n, roi):
    """One block of `n` frames at one exposure, written in time order.

    Returns the rows the session record needs.  A frame out of band is retaken;
    a frame that exhausts `MAX_RETAKES` is written anyway and flagged, because
    a gap in the series is worse than a frame `08` can exclude by its header.

    The cooler duty is recorded per frame, and it is not decoration.  The
    2026-08-31 smoke test held -10.0 C exactly while the duty climbed 64 -> 77%
    over half an hour and the pedestal moved 1.1 counts: the sensor was cold
    and the body was still equilibrating.  Sensor temperature alone cannot see
    that, so if the pedestal series turns out to track anything tonight, duty
    is the column `08` needs to test it against.  Recording it costs one USB
    read per frame and turns a confound into a measurement.
    """
    rows, retakes = [], 0
    for i in range(n):
        path = FRAMES / f"blk{index:02d}_{kind}_{i:03d}.fits"
        if path.exists():
            continue                                   # resuming; see the warning above
        for attempt in range(MAX_RETAKES + 1):
            mosaic, header = asi.capture(rig, exposure_s, imagetyp=kind.upper())
            if in_band(header) or attempt == MAX_RETAKES:
                break
            retakes += 1
        header["BLOCK"] = index
        duty = rig.get("CoolPowerPerc")
        F.write(path, mosaic, header)
        rows.append({"block": index, "kind": kind, "i": i, "file": path.name,
                     "exptime": header["EXPTIME"], "ccd_temp": header["CCD-TEMP"],
                     "duty_pct": duty, "roi_w": roi[2],
                     "date_obs": header["DATE-OBS"], "in_band": in_band(header)})
        time.sleep(FRAME_GAP_S)
    return rows, retakes


say(f"\n===== session 03 starting {dt.datetime.now():%Y-%m-%d %H:%M} =====")
say(f"gain {GAIN}, offset {OFFSET}, setpoint {SETPOINT_C} C, ambient "
    f"{AMBIENT_START_C} C, bias exposure {BIAS * 1e6:.0f} us")

In [ ]:
import csv

plan = schedule(BIAS)
t0, all_rows, total_retakes = time.monotonic(), [], 0
current_roi = ROI

for index, (kind, exposure_s, n, roi) in enumerate(plan):
    if roi != current_roi:
        say(f"  ROI -> {roi}")
        asi.set_roi(rig, *roi)
        asi.configure(rig, gain=GAIN, offset=OFFSET)   # re-asserted after an ROI change
        asi.capture(rig, exposure_s if exposure_s < 5 else 1.0)   # discard
        current_roi = roi

    started = time.monotonic()
    rows, retakes = capture_block(index, kind, exposure_s, n, roi)
    all_rows += rows
    total_retakes += retakes

    temps = [r["ccd_temp"] for r in rows if r["ccd_temp"] is not None]
    span = f"{min(temps)} to {max(temps)} C" if temps else "no reading"
    say(f"blk{index:02d} {kind:5s} {exposure_s:7.4g} s x {n:2d}  "
        f"{(time.monotonic() - started) / 60:5.1f} min  {span}"
        f"{f'  {retakes} retaken' if retakes else ''}"
        f"   [{(time.monotonic() - t0) / 3600:.2f} h elapsed]")

# Appended, not rewritten.  On a resumed night the rows from before the
# interruption are already in this file and describe frames that cannot be
# recaptured; opening it "w" would delete the record of half the session while
# leaving the frames themselves on disk.  The header goes in only when new.
FIELDS = ["block", "kind", "i", "file", "exptime", "ccd_temp", "duty_pct",
          "roi_w", "date_obs", "in_band"]
csv_path = DATA / "frames.csv"
fresh = not csv_path.exists()
with open(csv_path, "a", newline="", encoding="utf8") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    if fresh:
        w.writeheader()
    w.writerows(all_rows)

say(f"\n{len(all_rows)} frames written in {(time.monotonic() - t0) / 3600:.2f} h, "
    f"{total_retakes} retaken, {sum(not r['in_band'] for r in all_rows)} out of band")
if not all_rows:
    say("nothing was captured -- every block was already on disk")
log_file.close()

In [ ]:
AMBIENT_END_C = None      # <- record the real reading, then close the camera

print(f"ambient {AMBIENT_START_C} C -> {AMBIENT_END_C} C")
rig.close()
print("camera closed; the TEC is off and the sensor is warming")

## The analysis half

The rules this runs were fixed in `protocols/03-dark-bound.md` before the frames
existed, and one of them did not survive contact with the data. That is the
night's main result rather than a disappointment: **the camera's black level
occupies discrete states about one ADC count apart.** The step between them is
four times the whole dark signal a 600 s exposure can produce here, so rule 1 --
subtract a bias level from a dark level -- measures the state rather than the
sensor whenever the two frames are not in the same one.

The order below follows from that: find the states first, then do arithmetic
only on frames known to share one.

| publishes | what |
|---|---|
| `pedestal_series.csv` | every bias block against wall clock, with its state census |
| `dark_blocks.csv` | every dark block: level, excess, implied rate |
| `dark_constants.json` | the constants, each with provenance |

Two of the night's measurements are **immune to the offset state by
construction**, because a uniform level shift cancels in a spatial difference:
the dark-signal non-uniformity, and the glow gradient across the frame. Those
are the firm numbers. `D` itself is a bound, and the thing that bounds it is the
state, not the statistics.

In [ ]:
import datetime as dt
import json
import pathlib
import re
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, spatial as SP, stats as ST

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
FRAMES = ROOT / "data" / "session03" / "frames"
PLANES = list(SP.PLANES)

GAIN, OFFSET, SETPOINT_C = 250, 15, -10.0
G_E_PER_COUNT, G_ERR = 0.51303, 0.00097      # ptc_constants.json, gain 250
CONSTANTS = RESULTS / "dark_constants.json"
PEDESTAL_CSV = RESULTS / "pedestal_series.csv"
BLOCKS_CSV = RESULTS / "dark_blocks.csv"
MEASURED_ON = "2026-09-01"
NOTEBOOK = "07_dark.ipynb"

# One pass over the night.  Per frame, per plane, the mean in ADC counts --
# which is all the level analysis needs.  Pixel arrays are re-read later, and
# only for the three questions that need them.
rows = []
for path in sorted(FRAMES.glob("*.fits")):
    blk, kind = re.match(r"blk(\d+)_(\w+)_", path.name).groups()
    mosaic, hdr = F.read(path)
    row = {n: float(ST.to_adc(p).astype(np.float64).mean())
           for n, p in SP.split(mosaic).items()}
    row.update(block=int(blk), kind=kind, file=path.name,
               exptime=float(hdr["EXPTIME"]), date_obs=hdr["DATE-OBS"],
               ccd_temp=hdr["CCD-TEMP"], width=mosaic.shape[1])
    rows.append(row)

fr = pd.DataFrame(rows)
fr["t"] = pd.to_datetime(fr.date_obs)
fr["t_min"] = (fr.t - fr.t.min()).dt.total_seconds() / 60
fr["level"] = fr[PLANES].mean(axis=1)
fr["full_frame"] = fr.width > 2000
fr = fr.sort_values("t_min").reset_index(drop=True)

print(f"{len(fr)} frames over {fr.t_min.max():.0f} min")
print(f"temperature {fr.ccd_temp.min()} to {fr.ccd_temp.max()} C, "
      f"{(fr.ccd_temp.sub(SETPOINT_C).abs() > 0.5).sum()} frames out of band")
print(fr.groupby(["kind", "exptime", "full_frame"]).size())

In [ ]:
# The offset state.  A frame is compared with the median of every frame that
# shares its kind, exposure and ROI -- the only frames whose level it has any
# right to equal.  The states are ~1 count apart and the read noise is 1.7
# counts on a single pixel but 0.001 counts on a plane mean, so on the mean the
# two states are separated by a thousand sigma and the threshold is a formality.
STATE_THRESHOLD = 0.5           # counts; the step is ~1.0 and the scatter ~0.01

peer = fr.groupby(["kind", "exptime", "full_frame"])["level"]
fr["peer_level"] = peer.transform("median")
fr["departure"] = fr.level - fr.peer_level
fr["anomalous"] = fr.departure.abs() > STATE_THRESHOLD

bias = fr[fr.kind == "bias"]
step = (bias.loc[bias.anomalous, "departure"].mean()
        - bias.loc[~bias.anomalous, "departure"].mean())
first = fr[fr.anomalous].t_min.min()

print(f"anomalous frames: {fr.anomalous.sum()} of {len(fr)} "
      f"({100 * fr.anomalous.mean():.1f}%)")
print(f"step between states: {step:+.4f} counts "
      f"= {step * G_E_PER_COUNT:+.4f} e-")
print(f"first appearance: {first:.0f} min into the run "
      f"(nothing before it in {(fr.t_min < first).sum()} frames)")
print(f"\nwithin-state scatter of a plane mean: "
      f"{fr.loc[~fr.anomalous].groupby("block")["level"].std().median():.4f} counts")
print("\nby block:")
census = (fr.groupby(["block", "kind", "exptime"])
            .agg(n=("level", "size"), anomalous=("anomalous", "sum"),
                 level=("level", "mean"), t_min=("t_min", "mean"))
            .reset_index())
print(census[census.anomalous > 0].to_string(index=False))

In [ ]:
# The pedestal series: one row per bias block, anomalous frames excluded.
clean = fr[~fr.anomalous]
ped = (clean[clean.kind == "bias"]
       .groupby(["block", "full_frame"])
       .agg(**{"t_min": ("t_min", "mean"), "level": ("level", "mean"),
               "n_clean": ("level", "size"), "sd": ("level", "std")},
            **{n: (n, "mean") for n in PLANES})
       .reset_index())
ped["n_anomalous"] = [int(fr[(fr.block == b) & fr.anomalous].shape[0])
                      for b in ped.block]

roi_ped = ped[~ped.full_frame].sort_values("t_min").reset_index(drop=True)
slope, icept = np.polyfit(roi_ped.t_min, roi_ped.level, 1)
resid = roi_ped.level - (slope * roi_ped.t_min + icept)
PEDESTAL_SCATTER = float(resid.std(ddof=1))

print(f"{len(roi_ped)} ROI bias blocks over {roi_ped.t_min.max():.0f} min")
print(f"  level             {roi_ped.level.mean():.4f} counts")
print(f"  peak-to-peak      {roi_ped.level.max() - roi_ped.level.min():.4f}")
print(f"  trend             {slope * 60:+.4f} counts/hour")
print(f"  residual scatter  {PEDESTAL_SCATTER:.4f} counts")
print(f"  session 01 bound  +/-{0.254 * 60:.1f} counts/hour (15 min of data)")

ped.round(5).to_csv(PEDESTAL_CSV, index=False)
print(f"\nwrote {PEDESTAL_CSV}")

In [ ]:
# Rule 1, on frames known to share a state.  The pedestal is interpolated in
# time between the bracketing blocks; rule 2 said to establish the systematic
# first, and the systematic turned out to be the state rather than the drift.
def pedestal_at(minutes, full_frame=False):
    src = ped[ped.full_frame == full_frame].sort_values("t_min")
    return {n: float(np.interp(minutes, src.t_min, src[n])) for n in PLANES}


dk = (clean[clean.kind == "dark"]
      .groupby(["block", "exptime", "full_frame"])
      .agg(**{"t_min": ("t_min", "mean"), "level": ("level", "mean"),
              "n_clean": ("level", "size")},
           **{n: (n, "mean") for n in PLANES})
      .reset_index())
dk["n_anomalous"] = [int(fr[(fr.block == b) & fr.anomalous].shape[0])
                     for b in dk.block]
base = [pedestal_at(t, ff) for t, ff in zip(dk.t_min, dk.full_frame)]
dk["pedestal"] = [float(np.mean(list(b.values()))) for b in base]
dk["excess"] = dk.level - dk.pedestal
dk["implied_e_per_s"] = dk.excess * G_E_PER_COUNT / dk.exptime

print(dk[["block", "exptime", "full_frame", "t_min", "level", "pedestal",
          "excess", "implied_e_per_s"]].round(4).to_string(index=False))

# A dark current is one number.  Read off each exposure separately it is not,
# and the spread across exposures is what the night can honestly bound.
roi_dk = dk[~dk.full_frame]
per_exp = roi_dk.groupby("exptime").agg(
    blocks=("excess", "size"), excess=("excess", "mean"),
    sd=("excess", "std"), rate=("implied_e_per_s", "mean")).reset_index()
print("\n" + per_exp.round(6).to_string(index=False))

# The bound comes from the LONGEST lever arm and nowhere else.  A fixed offset
# divided by an exposure looks like a rate, and the shorter the exposure the
# larger that fake rate: the 1 s block turns 0.04 counts of offset into
# 2e-2 e-/px/s, which says nothing about dark current and everything about
# dividing by one second.  Only the longest exposure turns a level error into
# a tight rate, so only the longest exposure bounds anything.
LEVER = float(roi_dk.exptime.max())
longest = roi_dk[roi_dk.exptime == LEVER]
D_BOUND = float(abs(longest.excess.mean()) * G_E_PER_COUNT / LEVER)
gap = float(longest.level.mean() - roi_dk[roi_dk.exptime == 300].level.mean())

print("\nimplied rate by exposure, e-/px/s:")
for _, r in per_exp.iterrows():
    note = "   <- the bound" if r.exptime == LEVER else (
        "   (lever arm too short to bound anything)" if r.exptime < 300 else "")
    print(f"  {r.exptime:6.0f} s  {r.rate:+.3e}{note}")
print("\nThe excess does not scale with exposure -- it changes sign between")
print("300 s and 600 s -- so it is an offset, not a rate, and what this night")
print("publishes is a bound:")
print(f"  |D| < {D_BOUND:.2e} e-/px/s at -10 C   (L14 inherited < 1e-2)")
print(f"\nwhat sets it:  the offset state, {abs(step):.3f} counts, worth "
      f"{abs(step) * G_E_PER_COUNT / LEVER:.2e} e-/px/s over {LEVER:.0f} s")
print("what does not: statistics, at 1.3e-06 e-/px/s")
print(f"\n600 s and 300 s levels differ by {gap:+.3f} counts against a state "
      f"step of {abs(step):.3f}:")
print("the two exposures appear to sit in different states, which is the open"
      " question this night raises and cannot answer")

dk.round(6).to_csv(BLOCKS_CSV, index=False)
print(f"\nwrote {BLOCKS_CSV}")

In [ ]:
# The three questions that need pixels, not levels.  Two of them difference
# one frame against another *in space*, so a uniform offset state cancels and
# they are the night's firm numbers.
def plane_of(name, path):
    mosaic, _ = F.read(path)
    return ST.to_adc(SP.split(mosaic)[name]).astype(np.float64)


def files_for(block, kind):
    return sorted(FRAMES.glob(f"blk{block:02d}_{kind}_*.fits"))


PLANE = "G1"
clean_files = set(clean.file)

# --- DSNU at 300 s: single-frame variance minus pair-difference variance ---
d3 = [FRAMES / f for b in range(5, 20, 2) for f in
      (p.name for p in files_for(b, "dark")) if f in clean_files]
stack = np.stack([plane_of(PLANE, f) for f in d3])
keep = stack.mean(axis=0) < np.percentile(stack.mean(axis=0), 99.9)
single_var = float(np.var(stack[0][keep], ddof=1))
pair_var = float(np.var((stack[0] - stack[1])[keep], ddof=1) / 2)
DSNU = float(np.sqrt(max(single_var - pair_var, 0.0)))
print(f"D3 stack: {len(d3)} frames at 300 s, {100 * keep.mean():.2f}% of px kept")
print(f"  single-frame var {single_var:.4f}, pair var {pair_var:.4f}")
print(f"  DSNU at 300 s    {DSNU:.4f} counts = {DSNU * G_E_PER_COUNT:.4f} e-")

sd1 = float(stack[0][keep].std(ddof=1))
print(f"\n  N   sd of mean    ideal   ratio")
stall = []
for N in (1, 2, 4, 8, 16, 32):
    sd = float(stack[:N].mean(axis=0)[keep].std(ddof=1))
    stall.append((N, sd))
    print(f"{N:4d} {sd:12.5f} {sd1 / np.sqrt(N):8.5f} {sd / (sd1 / np.sqrt(N)):7.3f}")
print(f"  the stack stalls at {stall[-1][1]:.3f} counts against a DSNU of "
      f"{DSNU:.3f} -- the same number, which is the pre-registered reading")

# --- eta_comb, on bias rather than darks: bias carries no DSNU (session 01) ---
bias_files = [FRAMES / f for f in
              clean[(clean.kind == "bias") & (~clean.full_frame)].file]
bstack = np.stack([plane_of(PLANE, f) for f in bias_files])
bkeep = bstack.mean(axis=0) < np.percentile(bstack.mean(axis=0), 99.9)
bsd1 = float(bstack[0][bkeep].std(ddof=1))
print(f"\nbias stack: {len(bias_files)} clean frames")
eta = {}
for N in (2, 4, 8, 16, 32, 64, 128):
    if N > len(bstack):
        break
    sd = float(bstack[:N].mean(axis=0)[bkeep].std(ddof=1))
    eta[N] = (bsd1 / np.sqrt(N)) / sd
    print(f"  N={N:4d}  eta = {eta[N]:.4f}")
BIAS_FPN = float(np.sqrt(max(
    bstack[:128].mean(axis=0)[bkeep].var(ddof=1) - bsd1 ** 2 / 128, 0.0)))
print(f"  floor {BIAS_FPN:.4f} counts of bias fixed pattern; session 01's "
      f"ratio 1.011 predicts {bsd1 * np.sqrt(1.011 ** 2 - 1):.4f}")

# --- glow: corner minus centre, immune to any uniform offset ---
ffd = np.stack([plane_of(PLANE, p) for p in files_for(30, "dark")]).mean(axis=0)
ffb = np.stack([plane_of(PLANE, p) for p in files_for(29, "bias")]).mean(axis=0)
diff = ffd - ffb
h, w = diff.shape


def patch_mean(a):
    return float(a[a < np.percentile(a, 99.9)].mean())


regions = {"centre": diff[h // 2 - 128:h // 2 + 128, w // 2 - 128:w // 2 + 128],
           "top_left": diff[:256, :256], "top_right": diff[:256, -256:],
           "bot_left": diff[-256:, :256], "bot_right": diff[-256:, -256:],
           "session_roi": diff[568 // 2:568 // 2 + 512, 1408 // 2:1408 // 2 + 512]}
glow = {k: patch_mean(v) for k, v in regions.items()}
GLOW_GRADIENT = max(glow[k] for k in glow if k != "centre"
                    and k != "session_roi") - glow["centre"]
print(f"\nfull-frame 600 s dark minus its own bias, {PLANE} plane:")
for k, v in glow.items():
    print(f"  {k:12s} {v:+.4f} counts")
print(f"  corner - centre {GLOW_GRADIENT:+.4f} counts "
      f"= {GLOW_GRADIENT * G_E_PER_COUNT / 600:.2e} e-/px/s of spatial variation")

In [ ]:
N_FRAMES = int(len(fr))


def prov(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": N_FRAMES, "measured_on": MEASURED_ON,
            "notebook": NOTEBOOK, "note": note}


constants = {
    "offset_state_step": prov(
        round(float(abs(step)), 4), "ADC counts", 0.01,
        "the camera's black level occupies discrete states this far apart, on "
        "every plane at once and uniformly across the frame.  Detected per "
        "frame as a departure of a plane mean from its peers (threshold 0.5 "
        "counts against a within-state scatter of ~0.01), so it is rejectable "
        "rather than merely present.  It refutes protocol rule 1 as written: "
        "dark-minus-bias measures the state whenever the two frames differ in "
        "it, and the step is 4x the dark signal a 600 s exposure produces"),
    "offset_state_incidence": prov(
        round(float(fr.anomalous.mean()), 5), "fraction of frames", None,
        f"{int(fr.anomalous.sum())} of {N_FRAMES} frames.  Zero in the first "
        f"{int((fr.t_min < first).sum())} frames and then intermittent, so it "
        "is not a property of the settings; it appeared "
        f"{first:.0f} min into a 5 h run at a fixed configuration"),
    "pedestal_stability": prov(
        round(PEDESTAL_SCATTER, 4), "ADC counts (residual sd about a linear trend)",
        round(float(roi_ped.level.std(ddof=1)), 4),
        f"{len(roi_ped)} interleaved bias blocks over {roi_ped.t_min.max():.0f} "
        f"min at gain {GAIN}, offset {OFFSET}, -10 C, anomalous frames excluded. "
        f"Trend {slope * 60:+.4f} counts/hour.  Session 01 bounded the rate at "
        "+/-0.254 counts/min over 15 min; this is the same question over the "
        "span an imaging night occupies, and it is ~250x tighter"),
    "dark_current_bound": prov(
        round(D_BOUND, 6), "e-/px/s at -10 C", None,
        "an upper bound, not a value, and never a signed one (L14).  The "
        "implied rate read off each exposure separately spans "
        f"{per_exp.rate.min():+.3e} to {per_exp.rate.max():+.3e} e-/px/s and "
        "disagrees in sign, which is what a dark current cannot do.  What "
        "limits it is the offset state, not statistics: the state step over a "
        f"600 s exposure is {abs(step) * G_E_PER_COUNT / 600:.2e} e-/px/s, "
        "while the statistical floor is 1.3e-6.  L14 inherited < 1e-2"),
    "dsnu_300s": prov(
        round(DSNU, 4), "ADC counts at 300 s",
        round(float(np.sqrt(pair_var / (2 * len(d3)))), 4),
        "single-frame spatial variance minus pair-difference variance, on the "
        f"{PLANE} plane of {len(d3)} clean 300 s darks with the hottest 0.1% of "
        "pixels masked.  Immune to the offset state, which is uniform and "
        "cancels in a spatial difference.  The 32-frame stack stalls at "
        f"{stall[-1][1]:.3f} counts against this {DSNU:.3f}: the stall is DSNU, "
        "which is what protocol rule 4 pre-registered rather than an "
        "eta_comb failure"),
    "eta_comb": prov(
        {str(k): round(float(v), 4) for k, v in eta.items()},
        "measured sd reduction against the ideal sqrt(N)", None,
        "measured on the BIAS stack, not the dark stack.  L15 required frames "
        "that need no registration; darks satisfy that and then stall on their "
        "own DSNU, which is a property of darks and not of combination.  Bias "
        "carries no dark-signal non-uniformity (session 01's "
        "bias_fixed_pattern_ratio 1.011), so it isolates the "
        "rejection-and-averaging half honestly.  Averaging is ideal to N~8 and "
        f"then approaches a fixed-pattern floor of {BIAS_FPN:.4f} counts.  This "
        "is an upper bound on the real loss: no registration, no resampling, "
        "and the sky half is unmeasured"),
    "glow_gradient": prov(
        round(GLOW_GRADIENT, 4), "ADC counts at 600 s (worst corner minus centre)",
        None,
        "median-combined full-frame 600 s darks against their own full-frame "
        "bias, so a uniform offset state cancels.  Per-region means are "
        + ", ".join(f"{k} {v:+.4f}" for k, v in glow.items()) +
        ".  The session ROI sits in the quiet part of the frame, which is why "
        "the ROI's own value is what the model uses (protocol rule 5)"),
    "setpoint_held": prov(
        [float(fr.ccd_temp.min()), float(fr.ccd_temp.max())], "C", 0.5,
        f"every one of {N_FRAMES} frames in band; 0 retaken, 0 out of band "
        "across 5.03 h"),
}

with open(CONSTANTS, "w", encoding="utf8") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS} with {len(constants)} constants, provenance on each")
for k, v in constants.items():
    print(f"  {k:26s} {v['value']}")

## What the night decided

| protocol asked | answer |
|---|---|
| is the slope distinguishable from zero? | **no, and not for the reason expected.** The limit is a discrete offset state, not drift and not statistics. `D` leaves `sigma^2` in the model as a bound |
| does glow reach the ROI? | there is a gradient, worst corner to centre, and **the session ROI sits in the quiet part** |
| does the stack track sqrt(N)? | **no, and the stall is DSNU** -- the reading rule 4 pre-registered. `eta_comb` moved to the bias stack, where it belongs |

**What this costs the model.** `D` stays out of `sigma^2` at any exposure this
project will use: even the bound is four orders of magnitude below the sky rate
L32 predicts. The dark term is settled, and settled as absent.

**What it costs the method.** Any future measurement that subtracts a bias level
from a dark level at better than one count must first classify the offset state.
That is cheap -- a plane mean separates the states by a thousand sigma -- but it
is not optional, and it applies to the archive as much as to the bench.

**What is still open.** Why the state appears at all, and why it started 161
minutes into a five-hour run at a fixed configuration. Nothing here explains it,
and this night cannot: it was not designed to vary anything that might cause it.
That is a `DECISIONS` item and a candidate session, not a gap in this result.